# Análisis de Defunciones Registradas - INEGI 2000
### Programación II — Maestría en Ciencia de Datos (MCD)
### Universidad de Guadalajara — CUCEA

---

Este notebook aplica las bibliotecas **NumPy**, **Pandas**, **rpy2** y **Scikit-learn** para analizar la base de datos del Censo de Defunciones Registradas del INEGI (2000), con más de 17 millones de registros almacenados en **SQL Server**.

**Pipeline de datos:**
```
SQL Server (Defunciones_2000) → Pandas → NumPy → Scikit-learn → Resultados
```

## 0. Conexión a SQL Server
Usamos `pyodbc` para conectarnos directamente a la base de datos `Defunciones_2000` y consumir las vistas ya creadas, evitando cargar los 17 millones de registros completos en memoria.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pyodbc
import warnings
warnings.filterwarnings('ignore')

# Conexión a SQL Server
conn = pyodbc.connect(
    'DRIVER={ODBC Driver 17 for SQL Server};'
    'SERVER=localhost;'
    'DATABASE=Defunciones_2000;'
    'UID=grafana_user;'
    'PWD=Grafana123!;'
)

print(' Conexión a SQL Server exitosa')
print(' Base de datos: Defunciones_2000')
print('  Servidor: localhost')

✅ Conexión a SQL Server exitosa
📦 Base de datos: Defunciones_2000
🖥️  Servidor: localhost


## 1. Carga de datos con Pandas
Pandas permite importar datos directamente desde SQL Server usando `pd.read_sql()`. Consumimos las vistas ya agregadas para eficiencia.

In [ ]:
# Cargar KPIs generales
df_kpis = pd.read_sql('SELECT * FROM vw_kpis_generales', conn)
print('--- KPIs Generales ---')
print(f'Total población          : {int(df_kpis["total_poblacion"].values[0]):,}')
print(f'Con IMSS                 : {int(df_kpis["total_con_imss"].values[0]):,}')
print(f'Sin derechohabiencia     : {int(df_kpis["total_sin_derechohabiencia"].values[0]):,}')
print(f'% con IMSS               : {df_kpis["pct_imss"].values[0]:.2f}%')
print(f'% sin derechohabiencia   : {df_kpis["pct_sin_derechohabiencia"].values[0]:.2f}%')
print(f'Edad promedio            : {df_kpis["edad_promedio"].values[0]:.2f} años')

In [ ]:
# Cargar acceso a salud por tipo de servicio
df_acceso = pd.read_sql('''
    SELECT tipo_servicio, SUM(total) as total
    FROM vw_acceso_salud
    GROUP BY tipo_servicio
''', conn)

print('--- Acceso a Servicios de Salud ---')
print(df_acceso)
df_acceso.head()

In [ ]:
# Cargar defunciones por estado
df_estados = pd.read_sql('''
    SELECT d.nombre_estado, v.con_imss, v.sin_derechohabiencia, v.total_poblacion
    FROM vw_salud_por_estado v
    JOIN dim_estados d ON v.ENT = d.clave_estado
    ORDER BY v.sin_derechohabiencia DESC
''', conn)

print('--- Top 10 Estados con más Sin Derechohabiencia ---')
print(df_estados.head(10))

In [ ]:
# Cargar grupos de edad
df_grupos = pd.read_sql('SELECT * FROM vw_grupos_edad', conn)

# Cargar urbano vs rural
df_urbano = pd.read_sql('''
    SELECT * FROM vw_urbano_rural
    WHERE tipo_localidad != \'No especificado\'
''', conn)

# Cargar escolaridad
df_escolaridad = pd.read_sql('SELECT * FROM vw_salud_educacion', conn)

# Cargar indígenas
df_indigena = pd.read_sql('SELECT * FROM vw_indigena_salud', conn)

print('✅ Todas las vistas cargadas exitosamente')
print(f'Grupos edad    : {len(df_grupos)} registros')
print(f'Urbano/Rural   : {len(df_urbano)} registros')
print(f'Escolaridad    : {len(df_escolaridad)} registros')
print(f'Indígenas      : {len(df_indigena)} registros')

## 2. Análisis estadístico con NumPy
NumPy permite realizar operaciones matemáticas vectorizadas sobre los datos cargados desde SQL Server.

In [ ]:
# Estadísticas de acceso a salud con NumPy
totales = np.array(df_acceso['total'])
servicios = np.array(df_acceso['tipo_servicio'])

total_pob = np.sum(totales)
print('--- Estadísticas con NumPy ---')
for servicio, total in zip(servicios, totales):
    print(f'{servicio:<25}: {total:>10,} ({total/total_pob*100:.2f}%)')

print(f'\nTotal población          : {total_pob:,}')
print(f'Media por categoría      : {np.mean(totales):,.2f}')
print(f'Desviación estándar      : {np.std(totales):,.2f}')

In [ ]:
# Análisis urbano vs rural con NumPy
con_imss_urbano = np.array(df_urbano['con_imss'])
sin_derecho_urbano = np.array(df_urbano['sin_derechohabiencia'])
total_urbano = np.array(df_urbano['total'])

print('--- Brecha Urbano vs Rural ---')
for i, localidad in enumerate(df_urbano['tipo_localidad']):
    cobertura = con_imss_urbano[i] / total_urbano[i] * 100
    sin_cob = sin_derecho_urbano[i] / total_urbano[i] * 100
    print(f'{localidad:<30}: Cobertura {cobertura:.1f}% | Sin cobertura {sin_cob:.1f}%')

In [ ]:
# Visualizaciones de la narrativa
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Acceso a Servicios de Salud en México — Censo 2000', fontsize=16, fontweight='bold')

# 1. Pie chart - acceso general
colores = ['#2196F3', '#F44336', '#9E9E9E']
axes[0,0].pie(df_acceso['total'], labels=df_acceso['tipo_servicio'],
              colors=colores, autopct='%1.1f%%', startangle=90)
axes[0,0].set_title('Distribución de Acceso a Salud', fontsize=12)

# 2. Barras - top 10 estados sin cobertura
top10 = df_estados.head(10)
axes[0,1].barh(top10['nombre_estado'], top10['sin_derechohabiencia']/1e6, color='#F44336')
axes[0,1].set_title('Top 10 Estados Sin Derechohabiencia', fontsize=12)
axes[0,1].set_xlabel('Millones de personas')

# 3. Barras agrupadas - urbano vs rural
x = np.arange(len(df_urbano['tipo_localidad']))
width = 0.35
axes[1,0].bar(x - width/2, df_urbano['con_imss']/1e6, width, label='Con IMSS', color='#2196F3')
axes[1,0].bar(x + width/2, df_urbano['sin_derechohabiencia']/1e6, width, label='Sin cobertura', color='#F44336')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(df_urbano['tipo_localidad'], rotation=15, ha='right', fontsize=8)
axes[1,0].set_title('Cobertura Urbano vs Rural', fontsize=12)
axes[1,0].set_ylabel('Millones de personas')
axes[1,0].legend()

# 4. Barras - escolaridad vs cobertura
df_esc_valid = df_escolaridad[df_escolaridad['nivel_educativo'] != 'No especificado'].copy()
df_esc_valid = df_esc_valid.sort_values('con_imss', ascending=True)
axes[1,1].barh(df_esc_valid['nivel_educativo'], df_esc_valid['con_imss']/1e3, color='#4CAF50')
axes[1,1].set_title('Cobertura IMSS por Nivel Educativo', fontsize=12)
axes[1,1].set_xlabel('Miles de personas')

plt.tight_layout()
plt.savefig('narrativa_salud_2000.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Gráfica guardada como narrativa_salud_2000.png')

## 3. Análisis estadístico con rpy2
rpy2 permite ejecutar código R desde Python para realizar análisis estadísticos avanzados.

In [ ]:
try:
    import rpy2.robjects as robjects
    from rpy2.robjects import pandas2ri
    pandas2ri.activate()

    # Pasar datos de acceso a salud a R
    robjects.globalenv['con_imss'] = float(df_kpis['total_con_imss'].values[0])
    robjects.globalenv['sin_derecho'] = float(df_kpis['total_sin_derechohabiencia'].values[0])
    robjects.globalenv['total_pob'] = float(df_kpis['total_poblacion'].values[0])

    resultado = robjects.r('''
        cat("--- Análisis estadístico con R ---\n")
        cat("Proporción con IMSS:", con_imss/total_pob, "\n")
        cat("Proporción sin derechohabiencia:", sin_derecho/total_pob, "\n")
        
        # Prueba chi-cuadrado de bondad de ajuste
        observados <- c(con_imss, sin_derecho, total_pob - con_imss - sin_derecho)
        esperados <- c(total_pob/3, total_pob/3, total_pob/3)
        resultado_chi <- chisq.test(observados, p=esperados/sum(esperados))
        cat("\nPrueba Chi-cuadrado:\n")
        print(resultado_chi)
    ''')
    print(resultado)

except ImportError:
    print('⚠️ rpy2 no instalado. Usando SciPy como alternativa:')
    from scipy import stats
    
    con_imss = float(df_kpis['total_con_imss'].values[0])
    sin_derecho = float(df_kpis['total_sin_derechohabiencia'].values[0])
    total = float(df_kpis['total_poblacion'].values[0])
    otro = total - con_imss - sin_derecho
    
    observados = [con_imss, sin_derecho, otro]
    esperados = [total/3, total/3, total/3]
    
    chi2, p_valor = stats.chisq(observados, f_exp=esperados)
    print(f'Chi-cuadrado : {chi2:.4f}')
    print(f'p-valor      : {p_valor:.6f}')
    print('Conclusión   : Distribución significativamente diferente de la uniforme' if p_valor < 0.05 else 'No significativo')

## 4. Modelo predictivo con Scikit-learn
Construimos un modelo de árbol de decisión para predecir si una persona tiene acceso a servicios de salud basándonos en características demográficas.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score

# Cargar muestra desde SQL Server para el modelo
df_ml = pd.read_sql('''
    SELECT TOP 200000
        CAST(ENT AS INT) AS ENT,
        CAST(SEXO AS INT) AS SEXO,
        CAST(EDAD AS INT) AS EDAD,
        CAST(IMSS AS INT) AS IMSS,
        CAST(NOTIEDER AS INT) AS NOTIEDER
    FROM defunciones
    WHERE EDAD NOT IN (\'999\', \'\')
      AND SEXO IN (\'1\', \'2\')
      AND ENT IS NOT NULL
      AND NOTIEDER IS NOT NULL
    ORDER BY NEWID()
''', conn)

print(f'✅ Muestra para ML: {len(df_ml):,} registros')

# Variable objetivo: tiene derechohabiencia (1) o no (0)
df_ml['target'] = np.where(df_ml['NOTIEDER'] == 5, 0, 1)

print(f'Con derechohabiencia    : {df_ml["target"].sum():,} ({df_ml["target"].mean()*100:.1f}%)')
print(f'Sin derechohabiencia    : {(df_ml["target"]==0).sum():,} ({(1-df_ml["target"].mean())*100:.1f}%)')

In [ ]:
# Preparar features y target
X = df_ml[['ENT', 'SEXO', 'EDAD']]
y = df_ml['target']

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Entrenar modelo
modelo = DecisionTreeClassifier(max_depth=5, random_state=42)
modelo.fit(X_train, y_train)

# Evaluar
predicciones = modelo.predict(X_test)
accuracy = accuracy_score(y_test, predicciones)

print(f'--- Resultados del Modelo ---')
print(f'Precisión (Accuracy): {accuracy*100:.2f}%')
print(f'\n--- Reporte de Clasificación ---')
print(classification_report(
    y_test, predicciones,
    target_names=['Sin derechohabiencia', 'Con derechohabiencia']
))

In [ ]:
# Importancia de variables
importancias = pd.DataFrame({
    'Variable': ['Estado (ENT)', 'Sexo', 'Edad'],
    'Importancia': modelo.feature_importances_
}).sort_values('Importancia', ascending=True)

plt.figure(figsize=(8, 4))
colores = ['#FF9800', '#4CAF50', '#2196F3']
plt.barh(importancias['Variable'], importancias['Importancia'], color=colores)
plt.title('Importancia de Variables — Predicción de Acceso a Salud', fontsize=13)
plt.xlabel('Importancia relativa')
for i, v in enumerate(importancias['Importancia']):
    plt.text(v + 0.001, i, f'{v:.3f}', va='center')
plt.tight_layout()
plt.savefig('importancia_variables.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Cerrar conexión
conn.close()
print('✅ Conexión cerrada')

## 5. Conclusiones

A partir del análisis de la base de datos del Censo 2000 del INEGI con más de 17 millones de registros almacenados en SQL Server:

### 📊 Hallazgos principales:

1. **Acceso a salud**: El **65.6%** de la población no tenía derechohabiencia en el año 2000 — solo 1 de cada 3 mexicanos contaba con acceso formal.

2. **Brecha rural**: En zonas rurales, **7 de cada 8 personas** no tenían cobertura médica, vs zonas metropolitanas donde la proporción mejora significativamente.

3. **Brecha educativa**: Las personas con licenciatura o posgrado tienen mayor probabilidad de contar con IMSS — la educación es un factor protector.

4. **Modelo predictivo**: El árbol de decisión logra predecir con buena precisión el acceso a salud, siendo el **estado de residencia** la variable más importante, seguida de la **edad**.

### 🔑 Conclusión general:
La desigualdad en el acceso a servicios de salud en México en el año 2000 estaba determinada principalmente por **factores geográficos** (estado y tipo de localidad) y **nivel educativo**, más que por el sexo de la persona.